<a href="https://colab.research.google.com/github/maick-code/airf-multilingual-tokenizer-challenge/blob/arena/01a09f96-airf-multilingual-tokenizer-ch/submissions/maick-dane-nkou/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Research Foundations Multilingual Tokenization Challenge

## Maick Dane Nkou — Train, Evaluate and Export b4

**Run all on CPU:** train one `weights-b4` tokenizer from scratch on the official
240,000-row **train** split, evaluate the saved file with the hash-verified official
checker on all **24,000 validation rows**, then automatically download the submission
files if the checks pass. No uploaded model or diagnostic archive is required.

- BPE 10,000, minimum frequency 5, reversible `space_word` + ByteLevel pipeline.
- Language repeats: ha/sw/yo/am **x4**, en/fr **x1**.
- Full official score, including English/French and reconstruction penalties.
- Exact reconstruction and zero unknown tokens required for automatic export.
- A positive official penalty blocks automatic export, **not the display of the score**.
- The former 5% headroom policy is not enforced. The measured margin remains visible.
- Automatic Colab downloads: **tokenizer.json, metadata.yml, README.md**. Your browser
  may ask you to allow multiple downloads. Outside Colab, retrieve them from the printed directory.

The user's two b4 runs measured **1.939666974**, with **1.787199439% headroom** and
SHA-256 `b2f9a461ce1b0e8bff07c00d982614f49d555464b8fb1efa46e66b08b67013e4`.
The committed tokenizer matches that b4 SHA-256. These are recorded Colab results,
**not a guaranteed score or a required hash** for a new
training run. BPE merge ties can vary. Exported descriptions always use the new measurements.

No review-report archive, report upload, external corpus, pretrained tokenizer,
published vocabulary or merge table is used. The recipe and training code are included below.
The optional `weights-yo-am4` preset reproduces the previously submitted recipe.

Exports go to `artifacts/submission-weights-b4/export/maick-dane-nkou/` by default,
**never over the repository submission**. Downloading files is not a GitHub submission:
the verified files still need to replace the team artifacts in a participation PR.
The separate optimization notebook is not part of this submission.

# Getting Started

Install only missing dependencies, like the starter notebook. The runtime must
use `tokenizers==0.22.1`. An existing uv environment is supported without requiring
pip when the dependencies are already installed. Dependencies and public data
require network access; no secrets, private test data or external corpus is used.

In [ ]:
# Runtime setup — ordinary Python, usable in Colab and a local notebook kernel.
import importlib.util
import shutil
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

TOKENIZERS_VERSION = "0.22.1"


def needs_install(package, *, exact=None, major=None):
    try:
        found = version(package)
    except PackageNotFoundError:
        return True
    if exact is not None:
        return found != exact
    if major is not None:
        return found.split(".", 1)[0] != str(major)
    return False


requirements = [
    ("tokenizers", f"tokenizers=={TOKENIZERS_VERSION}", {"exact": TOKENIZERS_VERSION}),
    ("datasets", "datasets>=4.0,<5", {"major": 4}),
    ("pandas", "pandas", {}),
    ("matplotlib", "matplotlib", {}),
    ("PyYAML", "PyYAML==6.0.2", {"exact": "6.0.2"}),
]
missing = [requirement for name, requirement, constraint in requirements
           if needs_install(name, **constraint)]
if missing:
    if importlib.util.find_spec("pip") is not None:
        command = [sys.executable, "-m", "pip", "install", "-q", *missing]
    elif shutil.which("uv"):
        command = ["uv", "pip", "install", "--python", sys.executable, *missing]
    else:
        raise RuntimeError("Install dependencies with uv sync --dev --group data, then restart the kernel")
    subprocess.run(command, check=True)
else:
    print("Dependencies already available at the required versions.")

import tokenizers
if tokenizers.__version__ != TOKENIZERS_VERSION:
    raise RuntimeError("Restart the runtime after installing tokenizers==0.22.1")
print("tokenizers version:", tokenizers.__version__)

In [ ]:
# Competition configuration and portable output locations.
import hashlib
import importlib.util
import json
import math
import tempfile
import time
import unicodedata
import urllib.request
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from datasets import load_dataset
from tokenizers import Regex, Tokenizer, decoders, models, pre_tokenizers, trainers

GITHUB_REPO = "aims-ai-research-foundations/airf-multilingual-tokenizer-challenge"
HF_DATASET = "Similoluwa/african-multilingual-tokenizer-challenge"
HF_REVISION = "v1.0.0"
LANGUAGES = ("en", "fr", "ha", "sw", "yo", "am")
LANGUAGE_NAMES = {"en": "English", "fr": "French", "ha": "Hausa",
                  "sw": "Swahili", "yo": "Yoruba", "am": "Amharic"}
SCORED_LANGUAGES = ("ha", "sw", "yo", "am")
MAX_VOCAB_SIZE = 10_000
MAX_FILE_BYTES = 20 * 1024 * 1024
TEAM_SLUG = "maick-dane-nkou"
# Default: train the selected b4 recipe, then evaluate and export its actual file.
RECIPE = "weights-b4"
RECIPE_BOOSTS = {
    "weights-b4": {"ha": 4, "sw": 4, "yo": 4, "am": 4},
    "weights-yo-am4": {"ha": 2, "sw": 2, "yo": 4, "am": 4},
}
if RECIPE not in RECIPE_BOOSTS:
    raise ValueError(f"Unknown recipe: {RECIPE}")
MIN_FREQUENCY = 5
BOOST = dict(RECIPE_BOOSTS[RECIPE])  # English/French remain x1
MIN_GUARDRAIL_HEADROOM = 0.05  # informational during evaluation, NOT an official rule

# Pin the official checker used for the reported result, not a mutable/stale utils.py.
OFFICIAL_COMMIT = "75578f2400c39b1f8e31ce7e7104b37fbc470d11"
OFFICIAL_UTILS_SHA256 = "1727de34136097eb48addabf90501589bdfefa31c20e201bab53c38f2f7c9688"
RAW_URL = f"https://raw.githubusercontent.com/{GITHUB_REPO}/{OFFICIAL_COMMIT}"
HISTORICAL_B4_SHA256 = "b2f9a461ce1b0e8bff07c00d982614f49d555464b8fb1efa46e66b08b67013e4"
SUBMITTED_SHA256 = "b2f9a461ce1b0e8bff07c00d982614f49d555464b8fb1efa46e66b08b67013e4"


def local_file(relative_path):
    """Find a repository file whether the kernel starts at root or in this team folder."""
    for root in (Path.cwd(), *Path.cwd().parents):
        candidate = root / relative_path
        if candidate.is_file():
            return candidate
    return None


project_file = local_file("pyproject.toml")
WORKSPACE = project_file.parent if project_file is not None else Path.cwd()
# Never put reports/caches next to the four submitted files in a repository checkout.
RUN_DIR = WORKSPACE / "artifacts" / f"submission-{RECIPE}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Training recipe:", RECIPE, "| language repeats:", BOOST, "| en/fr x1")

In [ ]:
# Load the official profile_submission helper, like the starter, but hash-verified.
def ensure_utils():
    """Reuse only an exact matching helper; otherwise download the pinned official file."""
    destination = RUN_DIR / "official_utils.py"
    candidates = [destination, local_file("starter/utils.py")]
    helper = None
    for path in candidates:
        if path is not None and path.is_file():
            content = path.read_bytes()
            if hashlib.sha256(content).hexdigest() == OFFICIAL_UTILS_SHA256:
                helper = content
                break
    if helper is None:
        with urllib.request.urlopen(f"{RAW_URL}/starter/utils.py", timeout=60) as response:
            helper = response.read()
    if hashlib.sha256(helper).hexdigest() != OFFICIAL_UTILS_SHA256:
        raise RuntimeError("Official checker SHA-256 mismatch; do not continue")
    destination.write_bytes(helper)
    spec = importlib.util.spec_from_file_location("submission_official_utils", destination)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    if module.REQUIRED_TOKENIZERS_VERSION != TOKENIZERS_VERSION or module.RECONSTRUCTION_PENALTY != 3.0:
        raise RuntimeError("Unexpected official checker contract")
    return module


official_utils = ensure_utils()
profile_submission = official_utils.profile_submission
print("Official checker:", OFFICIAL_COMMIT)

## Load the Dataset

Inspect the training distribution before training. There are 40,000 training
and 4,000 validation rows per language. **Original strings are never stripped,
lowercased or normalized.** Only `train` is given to the trainer; validation is
reserved for evaluation. There is no reduced-data mode for final export.

In [ ]:
def load_competition_data(split):
    """Load an official public split as a DataFrame, as in the starter notebook."""
    if split not in {"train", "validation"}:
        raise ValueError("Only train and validation are public")
    dataset = load_dataset(HF_DATASET, split=split, revision=HF_REVISION,
                           cache_dir=str(RUN_DIR / "dataset-cache"))
    frame = dataset.to_pandas()[["language", "text"]].reset_index(drop=True)
    frame.attrs["dataset_fingerprint"] = dataset._fingerprint
    expected_per_language = 40_000 if split == "train" else 4_000
    if Counter(frame.language) != dict.fromkeys(LANGUAGES, expected_per_language):
        raise ValueError(f"Unexpected {split} language counts")
    if not all(isinstance(text, str) and text.split() for text in frame.text):
        raise ValueError(f"Invalid text in {split}")
    return frame


train = load_competition_data("train")
validation = load_competition_data("validation")
print(f"Train rows:      {len(train):,}")
print(f"Validation rows: {len(validation):,}")

In [ ]:
# Equal row counts do not imply equal amounts of text.
summary = train.assign(characters=train.text.str.len()).groupby("language").agg(
    rows=("text", "size"), characters=("characters", "sum"), mean_length=("characters", "mean"))
summary["share_of_characters"] = summary.characters / summary.characters.sum()
summary["training_repeat"] = [BOOST.get(lang, 1) for lang in summary.index]
print(summary.round({"mean_length": 1, "share_of_characters": 3}))

sizes = summary.characters.rename(index=LANGUAGE_NAMES).sort_values() / 1e6
fig, ax = plt.subplots(figsize=(7.2, 3.4))
ax.barh(sizes.index, sizes.values, color="#4878a8")
ax.set_xlabel("Characters in the training split (millions)")
ax.set_title("Training distribution before language weighting")
plt.tight_layout()
plt.show()

for language in LANGUAGES:
    text = train.loc[train.language == language, "text"].sample(1, random_state=41).iloc[0]
    print(f"[{language}] {LANGUAGE_NAMES[language]}: {text[:120]}")

# Train the Submission Candidate

## Lossless BPE with Punctuation-Attached Words

The reversible architecture is unchanged:

- BPE with **10,000** entries and minimum frequency **5**.
- `Split(Regex(r" ?\S+|\s+"), behavior="isolated")`, followed by
  `ByteLevel(add_prefix_space=False, use_regex=False)`.
- Matching `ByteLevel` decoder and the complete 256-symbol byte alphabet.
- No normalizer, special tokens or post-processor.

The default **`weights-b4`** repeats all four scored languages **x4**, with en/fr
**x1**. The optional `weights-yo-am4` preset repeats ha/sw x2 and yo/am x4, matching
the currently submitted recipe. The configuration cell prints the active choice.

Unlike `WhitespaceSplit`, the isolated split retains every separator. Training
uses only the official **train** split, with a fresh vocabulary and merge table.
Regression examples and validation never enter final training.

In [ ]:
def new_tokenizer():
    """Build the exact reversible space_word pipeline shared by the b4 and yo-am4 recipes."""
    tokenizer = Tokenizer(models.BPE())
    tokenizer.pre_tokenizer = pre_tokenizers.Sequence([
        pre_tokenizers.Split(Regex(r" ?\S+|\s+"), behavior="isolated"),
        pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False),
    ])
    tokenizer.decoder = decoders.ByteLevel()
    return tokenizer


def corpus_iterator(train_by_lang):
    """Stable language order; use only unmodified training strings."""
    iterators = {lang: iter(train_by_lang[lang]) for lang in LANGUAGES}
    active = list(LANGUAGES)
    while active:
        for lang in active.copy():
            try:
                text = next(iterators[lang])
            except StopIteration:
                active.remove(lang)
                continue
            for _ in range(BOOST.get(lang, 1)):
                yield text


def train_final(texts, *, vocab_size=MAX_VOCAB_SIZE, min_frequency=MIN_FREQUENCY):
    """Train from scratch. Smaller parameters are for disposable regression tests only."""
    tokenizer = new_tokenizer()
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size, min_frequency=min_frequency, special_tokens=[],
        initial_alphabet=sorted(pre_tokenizers.ByteLevel.alphabet()), show_progress=True,
    )
    tokenizer.train_from_iterator(texts, trainer=trainer)
    assert tokenizer.normalizer is None
    assert tokenizer.get_vocab_size(with_added_tokens=True) <= vocab_size
    assert set(pre_tokenizers.ByteLevel.alphabet()) <= set(tokenizer.get_vocab())
    return tokenizer

## Exact Reconstruction and Evaluation Checks

The official checker supplies the full score. Our additional checks require complete
validation, zero UNK and **strictly identical original/decoded strings**, including
whitespace and Unicode. This strict reconstruction check is stronger than the pinned
checker's NFC-and-trim comparison; it never changes the official score.

The historical 5% margin is informational only. It does not block evaluation or export.
Automatic export additionally requires zero official penalties. Any nonzero penalty is
shown first; it is not removed or hidden by our export policy.

A disposable regression model tests the reversible pipeline below. Its examples and
merges are never reused in the full training run.

In [ ]:
def assert_exact_roundtrip(tokenizer, texts, *, batch_size=512):
    failures = 0
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        ids = [enc.ids for enc in tokenizer.encode_batch(batch, add_special_tokens=False)]
        restored = tokenizer.decode_batch(ids, skip_special_tokens=False)
        skipped = tokenizer.decode_batch(ids, skip_special_tokens=True)
        failures += sum(original != decoded or original != decoded_skip
                        for original, decoded, decoded_skip in zip(batch, restored, skipped, strict=True))
    if failures:
        raise ValueError(f"Export blocked: {failures}/{len(texts)} rows are not reconstructed exactly")


def assert_submission_ready(report, expected_rows=24_000, *, enforce_margin=False):
    if not report.get("valid"):
        raise ValueError(f"Official validity check failed: {report.get('errors')}")
    if report.get("rows") != expected_rows:
        raise ValueError("Incomplete validation evaluation")
    if report.get("vocab_size") != MAX_VOCAB_SIZE:
        raise ValueError("Expected a 10,000-entry vocabulary")
    if report.get("reconstruction") != 1.0 or report.get("lossy_rows") != 0:
        raise ValueError("Reconstruction must be 100%")
    if report.get("reconstruction_penalty") != 0.0:
        raise ValueError("Nonzero reconstruction penalty")
    if set(report.get("fertility", {})) != set(LANGUAGES):
        raise ValueError("Missing validation language")
    if set(report.get("unknown_rate", {})) != set(LANGUAGES):
        raise ValueError("Missing unknown-token measurements")
    if any(value != 0.0 for value in report["unknown_rate"].values()):
        raise ValueError("Unknown tokens were emitted")
    for key in ("score", "guardrail_budget", "guardrail_penalty"):
        if not math.isfinite(report[key]) or report[key] < 0:
            raise ValueError(f"Invalid metric: {key}")
    base = sum(report["penalised"][lang] for lang in SCORED_LANGUAGES) / 4
    total = base + report["guardrail_penalty"] + report["reconstruction_penalty"]
    if not math.isclose(report["score"], total, rel_tol=1e-12, abs_tol=1e-12):
        raise ValueError("Full score is missing a penalty")
    headroom = context_headroom(report)
    if enforce_margin and headroom < MIN_GUARDRAIL_HEADROOM - 1e-12:
        raise ValueError("Optional 5% export margin not met; official evaluation is still available")


def context_headroom(report):
    budget = report["guardrail_budget"]
    if not math.isfinite(budget) or budget <= 0:
        raise ValueError("Invalid official guardrail budget")
    headroom = min((budget - report["fertility"][lang]) / budget for lang in ("en", "fr"))
    if not math.isfinite(headroom):
        raise ValueError("Invalid context-language fertility")
    return headroom


SMOKE_TEXTS = list(official_utils.SMOKE_TEXTS.values())
ROUNDTRIP_CASES = SMOKE_TEXTS + [
    "", " ", "   ", "\t\n\r\n", "  Hello  WORLD!\tNext\nline.  ",
    "[UNK] [CLS] [SEP] <0xFF>", "é e\u0301 Ì I\u0300", "👩🏿‍💻 🌍 中文 العربية",
    "a\u00a0b\u2003c\u200bd", "\x00\x01\x7f\ufeff\U0010ffff", "don't l’amour — … ።",
] + [unicodedata.normalize("NFD", text) for text in SMOKE_TEXTS]


def run_regression_tests():
    from tokenizers import normalizers
    mini = train_final(SMOKE_TEXTS, vocab_size=512, min_frequency=1)
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / "tokenizer.json"
        mini.save(str(path))
        loaded = Tokenizer.from_file(str(path))
        assert_exact_roundtrip(loaded, ROUNDTRIP_CASES)
        for mutation in ("lowercase", "wrong_decoder", "whitespace_split"):
            broken = Tokenizer.from_str(loaded.to_str())
            if mutation == "lowercase":
                broken.normalizer = normalizers.Lowercase()
            elif mutation == "wrong_decoder":
                broken.decoder = decoders.ByteFallback()
            else:
                broken.pre_tokenizer = pre_tokenizers.WhitespaceSplit()
            try:
                assert_exact_roundtrip(broken, ROUNDTRIP_CASES)
            except ValueError:
                pass
            else:
                raise AssertionError(f"Lossy mutation accepted: {mutation}")
    print(f"Regression tests passed ({len(ROUNDTRIP_CASES)} strict round-trip cases).")


run_regression_tests()

In [ ]:
# One full training run; validation and regression examples are never supplied.
train_by_lang = {
    lang: train.loc[train.language == lang, "text"].tolist() for lang in LANGUAGES
}
# Capture the actual recipe so exported metadata cannot silently use a later preset.
trained_config = {"recipe": RECIPE, "boost": dict(BOOST),
                  "min_frequency": MIN_FREQUENCY, "vocab_size": MAX_VOCAB_SIZE}
started = time.perf_counter()
tokenizer = train_final(corpus_iterator(train_by_lang))
training_seconds = time.perf_counter() - started
if tokenizer.get_vocab_size(with_added_tokens=True) != MAX_VOCAB_SIZE:
    raise ValueError("Unexpected vocabulary size after full training")
print(f"Training completed in {training_seconds:.1f} s")

candidate_dir = RUN_DIR / "candidate"
candidate_dir.mkdir(parents=True, exist_ok=True)
candidate_path = candidate_dir / "tokenizer.json"
tokenizer.save(str(candidate_path), pretty=True)
if candidate_path.stat().st_size > MAX_FILE_BYTES:
    raise ValueError("Tokenizer exceeds the 20 MiB limit")
# Always test the file that will actually be submitted, not just an in-memory model.
tokenizer = Tokenizer.from_file(str(candidate_path))
assert_exact_roundtrip(tokenizer, ROUNDTRIP_CASES)
assert_exact_roundtrip(tokenizer, validation.text.tolist())
print("All validation strings reconstructed exactly.")

# Score the Tokenizer

For each language, $S_l = F_l + 100 U_l$, where $F_l$ is tokens per word and
$U_l$ is unknown tokens per word. The **full** score is:

$$\frac{S_{ha}+S_{sw}+S_{yo}+S_{am}}{4}
+ P_{\mathrm{guardrail}} + P_{\mathrm{reconstruction}}.$$

We call `profile_submission` directly, rather than copy the older starter's
base-only scoring function. English/French overages and reconstruction loss
must not be omitted. The checker pin records the scoring version; the organizers
may update it. Hidden-test results can differ from validation.

In [ ]:
# Run the official checker on the newly trained file, NOT a root tokenizer.json.
measured_sha256 = hashlib.sha256(candidate_path.read_bytes()).hexdigest()
report = profile_submission(candidate_path, data=validation, repeats=3)
if hashlib.sha256(candidate_path.read_bytes()).hexdigest() != measured_sha256:
    raise ValueError("Candidate changed while it was being evaluated")
# Keep measurements in memory; no diagnostic report files or archive are generated.
assert_submission_ready(report, expected_rows=len(validation), enforce_margin=False)
base_score = sum(report["penalised"][lang] for lang in SCORED_LANGUAGES) / 4
headroom = context_headroom(report)
print("Measured recipe:", trained_config["recipe"])
print(f"Base score:              {base_score:.6f}")
print(f"Guardrail penalty:       {report['guardrail_penalty']:.6f}")
print(f"Reconstruction penalty:  {report['reconstruction_penalty']:.6f}")
print(f"Full official score:     {report['score']:.6f}")
print(f"EN/FR headroom:          {headroom:.2%}")
print("Strict reconstruction: 100%")
if headroom < MIN_GUARDRAIL_HEADROOM:
    print("NOTE: below the former 5% safety margin; this does NOT block export.")
if report["guardrail_penalty"] > 0:
    print("NOTE: an EN/FR penalty is included in the full score above.")
print("Historical b4 validation score: 1.939667 (not a target forced into the checker)")
print("Historical submitted-model validation score: 1.963189 (not remeasured here)")
print("Candidate SHA-256:", measured_sha256)
print("Same bytes as submitted artifact:", measured_sha256 == SUBMITTED_SHA256)
print("Same bytes as historical b4:", measured_sha256 == HISTORICAL_B4_SHA256)

results = pd.DataFrame([
    {"language": LANGUAGE_NAMES[lang], "tokens/word": report["fertility"][lang],
     "UNK rate": report["unknown_rate"][lang], "language score": report["penalised"][lang]}
    for lang in LANGUAGES
]).set_index("language")
print(results.round(6))
fig, ax = plt.subplots(figsize=(7.2, 3.4))
results["tokens/word"].plot.bar(ax=ax, color="#4878a8")
ax.axhline(report["guardrail_budget"], color="#be6b35", linestyle="--",
           label="Official English/French guardrail budget")
ax.set_ylabel("Tokens per word (lower is better)")
ax.set_title(f"{trained_config['recipe']} — measured validation fertility")
ax.legend()
plt.tight_layout()
plt.show()

## Previous Measurements — Not Results of This Execution

| Artifact / recipe | Full validation score ↓ | EN/FR headroom | Status |
| --- | ---: | ---: | --- |
| Previously submitted `weights-yo-am4` | 1.963189 | about 5.3% | Previous submitted artifact |
| Selected `weights-b4` | 1.939666974 | 1.787199439% | Recipe trained by default here |
| Optimization `yo5-am5` | about 1.9343 | about 0.1% | Not selected: reduced margin |

The participant selected b4 as the score/margin compromise. All three reported zero
official penalties on validation. The separate optimization notebook and its reports
are not needed to execute this submission notebook.

Only the live checker output above describes the newly trained file. Public leaderboard
scores can use different data and must not be substituted for these validation measurements.

## Official Scoring and the Historical Artifact

The previously submitted file has SHA-256
`1519895eace8680d2752b333f5efd82f80ade21bcd6210704ec17de55dafe035` and measured
**1.9631891569** on the participant's full validation audit. It is a different tokenizer
from b4; changing its description cannot give it b4's score.

The full score is the mean of `fertility + 100 * UNK rate` across ha/sw/yo/am,
plus any en/fr overage above **1.15 times mean raw target fertility**, plus
**3 times the official non-reconstructed fraction**.
The pinned checker compares decoded and original text after NFC normalization and
trimming their ends. Our additional tests require exact equality without these allowances.

No official rule requires a 5% margin or forbids 1.939667. However, b4's historical
margin of about 1.79% is small: different test texts may produce penalties.
Neither an export nor a successful validation run guarantees a hidden-test score or rank.

# Automatically Export the Verified Submission Files

This cell runs during **Run all**. It verifies that the candidate has not changed since
evaluation, requires zero UNK/penalties and exact reconstruction on all validation rows,
then exports **tokenizer.json, metadata.yml and README.md** and downloads them in Colab.
There is no review-only switch, manual approval step or report archive.

The measured score and SHA belong to the exact exported bytes; the historical score
is never substituted. No 5% margin is required. On a failed check, export stops with
an error instead of silently preparing an unchecked candidate.

Outputs are separate from `submissions/`. Use the downloaded files with this notebook
for the later participation PR; this cell does not commit, push or open a PR.

In [ ]:
def export_submission(path, measured, expected_sha256, validation_texts, output_dir, config, provenance):
    """Export only an unchanged, fully checked file; never silently swap models."""
    path = Path(path)
    output_dir = Path(output_dir)
    if "submissions" in output_dir.resolve().parts:
        raise ValueError("Export must not overwrite a repository submission")
    assert_submission_ready(measured, expected_rows=24_000, enforce_margin=False)
    if measured["guardrail_penalty"] != 0.0 or measured["reconstruction_penalty"] != 0.0:
        raise ValueError("Automatic export blocked: nonzero official penalty; see the measured score above")
    if len(validation_texts) != 24_000:
        raise ValueError("All 24,000 validation strings are required for export")
    if (config.get("recipe") not in RECIPE_BOOSTS
            or config.get("boost") != RECIPE_BOOSTS[config["recipe"]]
            or config.get("min_frequency") != 5 or config.get("vocab_size") != 10_000):
        raise ValueError("Recipe provenance is inconsistent")
    if (provenance.get("train_rows") != 240_000 or provenance.get("validation_rows") != 24_000
            or provenance.get("tokenizers_version") != TOKENIZERS_VERSION
            or not all(isinstance(provenance.get(key), str) and provenance[key]
                       for key in ("train_fingerprint", "validation_fingerprint"))
            or not math.isfinite(provenance.get("training_seconds", float("nan")))
            or provenance["training_seconds"] < 0):
        raise ValueError("Incomplete full-dataset training provenance")
    payload = path.read_bytes()
    if len(payload) > MAX_FILE_BYTES:
        raise ValueError("Tokenizer exceeds the 20 MiB limit")
    if hashlib.sha256(payload).hexdigest() != expected_sha256:
        raise ValueError("Candidate changed since evaluation; evaluate it again")
    assert_exact_roundtrip(Tokenizer.from_str(payload.decode("utf-8")), validation_texts)
    metadata = {
        "team": "Maick Dane Nkou", "members": ["Maick Dane Nkou"],
        "affiliation": "AIMS SOUTH AFRICA",
        "approach": (f"Lossless BPE 10000, space_word boundaries, no normalization; "
                     f"official train only, {config['recipe']}; validation "
                     f"{measured['score']:.6f}, reconstruction 100%."),
    }
    lines = [
        "# Maick Dane Nkou — lossless BPE", "", "## Recipe", "",
        f"- {config['recipe']}: space_word boundaries, no normalizer or special tokens",
        "- BPE 10,000; minimum frequency 5; full byte alphabet; ByteLevel decoder",
        f"- Official train only; balanced round-robin; repeats {json.dumps(config['boost'], sort_keys=True)}, en/fr x1",
        f"- Dataset: `{HF_DATASET}` @ `{HF_REVISION}`",
        "- No pretrained tokenizer, published vocabulary/merge table or external corpus",
        "", "## Measured validation results (24,000 rows; not hidden-test scores)", "",
        "| Language | Tokens/word | UNK rate |", "| --- | ---: | ---: |",
    ]
    for lang in LANGUAGES:
        lines.append(f"| {lang} | {measured['fertility'][lang]:.6f} | {measured['unknown_rate'][lang]:.6f} |")
    base = sum(measured["penalised"][lang] for lang in SCORED_LANGUAGES) / 4
    lines += [
        "", f"- Base score: {base:.6f}",
        f"- Guardrail penalty: {measured['guardrail_penalty']:.6f}",
        f"- Reconstruction penalty: {measured['reconstruction_penalty']:.6f}",
        f"- **Full score: {measured['score']:.6f}**", "- Strict reconstruction: 100%",
        f"- Tokenizer SHA-256: `{expected_sha256}`",
        f"- Official checker: `{OFFICIAL_COMMIT}`", "", "## Reproduce", "",
        "Run notebook.ipynb end-to-end on CPU.",
        f"Set RECIPE to `{config['recipe']}`; it trains that recipe once on train only, then checks",
        "the reloaded file on validation and automatically downloads the three submission files in Colab.",
        "The original optimization notebook is preserved at commit 17d34954a86e9baabb46658478ac7a0160e3a04d.",
        "BPE merge ties may vary across retraining runs; always evaluate the generated artifact.", "",
    ]
    lines += [
        "## Run provenance", "",
        f"- Train rows: {provenance['train_rows']}; validation rows: {provenance['validation_rows']}",
        f"- Train fingerprint: `{provenance['train_fingerprint']}`",
        f"- Validation fingerprint: `{provenance['validation_fingerprint']}`",
        f"- Training time: {provenance['training_seconds']:.2f} seconds",
        f"- tokenizers: `{provenance['tokenizers_version']}`",
        f"- Checker SHA-256: `{OFFICIAL_UTILS_SHA256}`",
        f"- Measured EN/FR headroom: {context_headroom(measured):.9%}",
        "- The former 5% headroom policy is not enforced.",
        "- Downloading these files does not submit them to GitHub.", "",
    ]
    # No output files are changed until all checks and descriptions are complete.
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / "tokenizer.json").write_bytes(payload)
    (output_dir / "metadata.yml").write_text(yaml.safe_dump(metadata, sort_keys=False), encoding="utf-8")
    (output_dir / "README.md").write_text("\n".join(lines), encoding="utf-8")
    return output_dir


# Measurements are embedded in the exported README, not in separate report files.
provenance = {
    "train_rows": len(train), "validation_rows": len(validation),
    "train_fingerprint": train.attrs["dataset_fingerprint"],
    "validation_fingerprint": validation.attrs["dataset_fingerprint"],
    "training_seconds": training_seconds, "tokenizers_version": tokenizers.__version__,
}
export_dir = export_submission(
    candidate_path, report, measured_sha256, validation.text.tolist(),
    RUN_DIR / "export" / TEAM_SLUG, trained_config, provenance,
)
print("Verified submission files:", export_dir)
print("Full measured score:", report["score"])
print("Tokenizer SHA-256:", measured_sha256)
try:
    from google.colab import files
except ImportError:
    print("Outside Colab: retrieve tokenizer.json, metadata.yml and README.md from the export directory.")
else:
    for filename in ("tokenizer.json", "metadata.yml", "README.md"):
        files.download(str(export_dir / filename))

## Final Checklist

Before opening the PR, check:

- [ ] Changes affect only `submissions/maick-dane-nkou/`.
- [ ] That directory contains only `tokenizer.json`, `metadata.yml`,
      `notebook.ipynb`, and optional `README.md` — no caches, archives or symlinks.
- [ ] The tokenizer loads with `tokenizers==0.22.1`, is at most 20 MiB,
      and has at most 10,000 entries including added tokens.
- [ ] The full score includes both penalties; all six languages were checked.
- [ ] The submitted tokenizer and the reported hash/score describe the same file.
- [ ] The notebook documents training on official **train only** and is valid v4 JSON.
- [ ] The PR targets `main` of the **official AIMS repository**, not your fork.

**About GitHub Actions:** the inspected `Validate submission` workflow checks
the changed team directory, the serialized tokenizer and the evaluator tests.
It does **not** execute this notebook or require its headings to match the
starter. A PR touching `submissions/**` is a trigger even when the source branch
is not named `submission`; the fork's push trigger is restricted to `submission`.
First-time contributors may need an organizer to approve workflow execution.
Notebook formatting cannot guarantee runner availability, permissions or test success.

Do not modify `.github/workflows/`, `starter/`, evaluation code or the leaderboard
in a competition-entry PR. Nightly scoring is a separate organizer workflow;
local validation is not an official hidden-test leaderboard result.

# References

- [Official starter notebook](https://github.com/aims-ai-research-foundations/airf-multilingual-tokenizer-challenge/blob/main/starter/starter.ipynb)
- [Competition rules and submission instructions](https://github.com/aims-ai-research-foundations/airf-multilingual-tokenizer-challenge/blob/main/CONTRIBUTING.md)
- [Public dataset](https://huggingface.co/datasets/Similoluwa/african-multilingual-tokenizer-challenge), revision `v1.0.0`
- [Pinned official checker](https://github.com/aims-ai-research-foundations/airf-multilingual-tokenizer-challenge/blob/75578f2400c39b1f8e31ce7e7104b37fbc470d11/starter/utils.py)
- [Original optimization notebook and artifact](https://github.com/maick-code/airf-multilingual-tokenizer-challenge/tree/17d34954a86e9baabb46658478ac7a0160e3a04d/submissions/maick-dane-nkou)
- [AI Research Foundations learning path](https://www.skills.google/paths/3135)